# CardioSafe AI - Notebook 10: the live demo (Gradio)

Launches the app with three modes: **ECG only**, **prescription only**, **ECG + prescription**. Also packages a ready-to-upload Hugging Face Space (`cardiosafe_space.zip`) so the demo can run without Colab.

What the demo shows for a case: the 12-lead ECG with a Grad-CAM overlay, calibrated class probabilities with conformal confidence, QTc status with its uncertainty band, the medications read from the prescription (editable text), and for each (ECG finding, drug) one of the three outcomes with the quoted FDA label sentence.

Runtime: GPU or CPU both work. Needs notebooks 03 to 09 done. Keep this notebook running while you present.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
WORK = '/content/drive/MyDrive/CardioSafe'; KB = WORK + '/kb'
sys.path.insert(0, WORK); os.makedirs(WORK + '/demo', exist_ok=True)
!apt-get -qq install -y tesseract-ocr > /dev/null 2>&1
!pip -q install gradio pytesseract rapidfuzz > /dev/null 2>&1
import gradio; print('gradio', gradio.__version__)

In [ ]:
%%writefile /content/drive/MyDrive/CardioSafe/cardiosafe_combine.py
"""Combined module: ECG findings x prescription -> evidence-backed relations with exactly three outcomes per (finding, drug):
  relationship_found       the FDA label text states the finding as an effect of the drug, or cautions/contraindicates the drug
                           in patients who have the finding (sentence, label id and link are attached)
  none_found               label retrieved, no statement about this finding in the available labels (NOT evidence of safety)
  insufficient_evidence    drug not recognised, no label retrieved, only weak mentions, or the ECG finding itself is unreliable
Nothing here is a diagnosis or a causal claim; every statement quotes its source."""
import cardiosafe_kb as K

STRONG = ('strong', 'moderate')
TOPIC_LABEL = {'qt': 'QT prolongation', 'bradycardia': 'slow heart rate (bradycardia)', 'av_block': 'conduction disturbance / heart block'}


def ecg_findings(ecg, hr_cut=50.0, hr_borderline=60.0, qt_unstable_sd_ms=25.0):
    """ecg: dict from ECGEngine.analyze. -> {topic: {'level': present|borderline|absent|unreliable, 'detail': str}}"""
    f = {}
    qtc, st = ecg['qtc_fridericia_ms'], ecg['qtc_status']
    base = f"QTc (Fridericia) estimated {qtc:.0f} ms (limit {ecg['qtc_limit_ms']:.0f} ms, uncertainty band +/-{ecg['qtc_band_ms']:.0f} ms)"
    if ecg.get('qt_seed_sd_ms', 0) > qt_unstable_sd_ms:
        f['qt'] = {'level': 'unreliable', 'detail': base + f"; the three QT models disagree by {ecg['qt_seed_sd_ms']:.0f} ms (SD), so the estimate is not trusted"}
    else:
        f['qt'] = {'level': {'prolonged': 'present', 'borderline': 'borderline', 'normal': 'absent'}[st], 'detail': base}
    hr = ecg['hr_bpm']
    lvl = 'present' if hr < hr_cut else 'borderline' if hr < hr_borderline else 'absent'
    f['bradycardia'] = {'level': lvl, 'detail': f'heart rate estimated {hr:.0f} bpm from the learned RR estimate'}
    cd = ecg['conformal'].get('CD', 'confident_negative')
    lvl = 'present' if cd == 'confident_positive' else 'borderline' if cd in ('uncertain', 'empty') and ecg['probs'].get('CD', 0) >= 0.3 else 'absent'
    f['av_block'] = {'level': lvl, 'detail': f"ECG classifier conduction-disturbance class p={ecg['probs'].get('CD', 0):.2f} (this class includes bundle-branch blocks; "
                                             'AV block / PR interval is not measured separately)'}
    return f


def _cite(e):
    return {'text': e['text'], 'section': e['group'], 'set_id': e.get('set_id'), 'url': e.get('url'), 'brand': (e.get('brand') or [''])[0] if isinstance(e.get('brand'), list) else e.get('brand')}


def assess(ecg, meds, fetch, ddi=None):
    """ecg: ECGEngine result; meds: extract_prescription output; fetch(ingredient) -> label record or None; ddi: optional K.DDI."""
    findings = ecg_findings(ecg)
    rows, notes, ingredients, nolabel = [], [], [], set()
    for m in meds:
        if m['status'] != 'ok' or not m['ingredients']:
            rows.append({'drug': m['name'], 'ingredient': None, 'topic': None, 'finding_level': None, 'outcome': 'insufficient_evidence',
                         'reason': f"drug name '{m['name']}' not recognised, so no label could be checked (status: {m['status']})", 'evidence': []})
            continue
        for ing in m['ingredients']:
            if ing not in ingredients:
                ingredients.append(ing)
    recs = {ing: fetch(ing) for ing in ingredients}
    for ing in ingredients:
        for topic, fd in findings.items():
            s = K.summarize_evidence(recs[ing], topic)
            eff, pc = s['by_role']['drug_effect'], s['by_role']['patient_condition']
            if fd['level'] in ('absent', 'unreliable'):
                if fd['level'] == 'absent' and (eff['tier'] in STRONG):
                    notes.append({'ingredient': ing, 'topic': topic, 'note': f"label lists {TOPIC_LABEL[topic]} as a possible effect (tier {eff['tier']}, "
                                  f"{eff['n_positive']}/{s['n_labels']} labels); no matching ECG finding in this tracing", 'evidence': [_cite(e) for e in eff['evidence'][:1]]})
                if fd['level'] == 'unreliable':
                    rows.append({'drug': ing, 'ingredient': ing, 'topic': topic, 'finding_level': 'unreliable', 'outcome': 'insufficient_evidence',
                                 'reason': 'the ECG-derived finding is unreliable: ' + fd['detail'], 'evidence': []})
                continue
            base = {'drug': ing, 'ingredient': ing, 'topic': topic, 'finding_level': fd['level'], 'finding_detail': fd['detail']}
            if s['tier'] == 'no_label':
                if ing not in nolabel:      # one row per ingredient, not one per finding
                    nolabel.add(ing)
                    rows.append({'drug': ing, 'ingredient': ing, 'topic': None, 'finding_level': None, 'outcome': 'insufficient_evidence',
                                 'reason': 'no FDA label could be retrieved for this ingredient, so the ECG findings above could not be checked against it', 'evidence': []})
                continue
            rels = []
            if eff['tier'] in STRONG:
                rels.append({'relation': 'drug_effect', 'tier': eff['tier'], 'n': eff['n_positive'], 'text': f"label lists {TOPIC_LABEL[topic]} as a possible effect of {ing}",
                             'evidence': [_cite(e) for e in eff['evidence']]})
            if pc['tier'] in STRONG:
                rels.append({'relation': 'patient_condition', 'tier': pc['tier'], 'n': pc['n_positive'],
                             'text': f"label advises caution or lists a contraindication for {ing} in patients with {TOPIC_LABEL[topic]}", 'evidence': [_cite(e) for e in pc['evidence']]})
            if rels:
                rows.append({**base, 'outcome': 'relationship_found', 'relations': rels, 'n_labels': s['n_labels'], 'evidence': [c for r in rels for c in r['evidence']],
                             'caveat': ('ECG finding is borderline: ' if fd['level'] == 'borderline' else '') + 'label statements show prominence in the label, not proof that this drug caused this ECG finding'})
            elif max(eff['tier'], pc['tier'], key=lambda t: K.TIER_RANK[t]) == 'weak' or s['tier'] in ('mention', 'weak'):
                rows.append({**base, 'outcome': 'insufficient_evidence', 'reason': 'only weak mentions found (overdose/pharmacology text or a bare mention)', 'evidence': [_cite(e) for e in s['evidence'][:1]]})
            else:
                rows.append({**base, 'outcome': 'none_found', 'n_labels': s['n_labels'],
                             'reason': f"no statement about {TOPIC_LABEL[topic]} in the {s['n_labels']} FDA label(s) retrieved; this is not evidence that the drug is safe", 'evidence': []})
    pairs = []
    qt_found = [r['ingredient'] for r in rows if r['topic'] == 'qt' and r['outcome'] == 'relationship_found' and any(x['relation'] == 'drug_effect' for x in r['relations'])]
    if findings['qt']['level'] in ('present', 'borderline') and len(set(qt_found)) >= 2:
        pairs.append({'type': 'multiple_qt_drugs', 'drugs': sorted(set(qt_found)), 'text': 'two or more prescribed drugs have QT prolongation statements in their labels'})
    if ddi is not None and len(ddi) and len(ingredients) >= 2:
        for r in ddi.check(ingredients):
            pairs.append({'type': 'ddinter', 'drugs': [r['drug_a'], r['drug_b']], 'level': r['level'], 'text': f"DDInter lists a {r['level']} interaction"})
    found = [r for r in rows if r['outcome'] == 'relationship_found']
    n_unrec = sum(r['outcome'] == 'insufficient_evidence' for r in rows)
    overall = 'attention' if found or pairs else 'insufficient' if n_unrec and not any(r['outcome'] == 'none_found' for r in rows) else 'none_found'
    return {'findings': findings, 'rows': rows, 'pairs': pairs, 'background_notes': notes, 'overall': overall}


DISCLAIMER = ('Research prototype for education. It is not a medical device and must not be used for diagnosis or to change treatment. '
              'ECG values are model estimates trained on PTB-XL (single dataset, no external validation); label statements come from FDA drug labels '
              'and show what a label says, not that a drug caused an ECG finding. "None found" never means "safe".')


def render_report(ecg, meds, result):
    """Plain-text/markdown report for the demo."""
    L = ['## ECG', f"Classifier (calibrated, conformal at 90%): " + ', '.join(f"{c} {ecg['probs'][c]:.2f} ({ecg['conformal'][c].replace('_', ' ')})" for c in ecg['probs']),
         f"QT {ecg['qt_ms']:.0f} ms, QTc (Fridericia) {ecg['qtc_fridericia_ms']:.0f} ms -> **{ecg['qtc_status']}** (limit {ecg['qtc_limit_ms']:.0f} ms, band +/-{ecg['qtc_band_ms']:.0f} ms); heart rate {ecg['hr_bpm']:.0f} bpm",
         '']
    if not meds:
        fnd = ecg_findings(ecg)
        L += ['## Findings relevant to medication safety'] + [f"- {TOPIC_LABEL[t]}: **{v['level']}** - {v['detail']}" for t, v in fnd.items()]
        L += ['', '_No prescription provided, so no medication check was done._', '', '> ' + DISCLAIMER]
        return '\n'.join(L)
    L.append('## Medications read from the prescription')
    for m in meds:
        tag = ', '.join(m['ingredients']) if m['status'] == 'ok' else f"UNRECOGNISED ({m['status']})"
        L.append(f"- {m['raw']}  ->  **{tag}**")
    L += ['', '## ECG finding x medication']
    titles = {'attention': 'Relationships found in drug labels - review recommended', 'none_found': 'No label relationship found for the findings present (not proof of safety)',
              'insufficient': 'Insufficient evidence for some or all items'}
    L.append(f"**{titles[result['overall']]}**")
    for r in result['rows']:
        head = f"- **{r['drug']}**" + (f" / {TOPIC_LABEL[r['topic']]} (ECG finding {r['finding_level']})" if r.get('topic') else '')
        if r['outcome'] == 'relationship_found':
            L.append(head + ': **relationship found**')
            for rel in r['relations']:
                L.append(f"    - {rel['text']} (label tier {rel['tier']}, in {rel['n']}/{r['n_labels']} labels)")
                for e in rel['evidence'][:1]:
                    L.append(f"      > \"{e['text'][:300]}\" [{e['section']}]" + (f" ({e['url']})" if e.get('url') else ''))
            L.append('    - ' + r['caveat'])
        elif r['outcome'] == 'none_found':
            L.append(head + ': none found in available sources - ' + r['reason'])
        else:
            L.append(head + ': **insufficient evidence** - ' + r['reason'])
    for p in result['pairs']:
        L.append(f"- Combination: {' + '.join(p['drugs'])}: {p['text']}")
    if result['background_notes']:
        L += ['', '## Other label notes (no matching ECG finding today)']
        L += [f"- {n['ingredient']}: {n['note']}" for n in result['background_notes']]
    L += ['', '> ' + DISCLAIMER]
    return '\n'.join(L)


def drug_only_report(meds, fetch):
    """Prescription-only mode (no ECG): what the FDA labels say about QT, slow heart rate and heart block for each drug."""
    L = ['## Medications read from the prescription']
    for m in meds:
        tag = ', '.join(m['ingredients']) if m['status'] == 'ok' else f"UNRECOGNISED ({m['status']}) - cannot be checked"
        L.append(f"- {m['raw']}  ->  **{tag}**")
    L += ['', '## What the FDA labels say about cardiac effects']
    seen = set()
    for m in meds:
        if m['status'] != 'ok':
            continue
        for ing in m['ingredients']:
            if ing in seen:
                continue
            seen.add(ing); rec = fetch(ing)
            if rec is None or not rec.get('labels'):
                L.append(f"- **{ing}**: insufficient evidence - no FDA label could be retrieved"); continue
            parts = []
            for topic in K.TOPICS:
                s = K.summarize_evidence(rec, topic)
                eff, pc = s['by_role']['drug_effect'], s['by_role']['patient_condition']
                if eff['tier'] in STRONG:
                    e = eff['evidence'][0]
                    parts.append(f"  - {TOPIC_LABEL[topic]}: label lists it as a possible effect (tier {eff['tier']}, {eff['n_positive']}/{s['n_labels']} labels)\n    > \"{e['text'][:260]}\"" + (f" ({e['url']})" if e.get('url') else ''))
                if pc['tier'] in STRONG:
                    e = pc['evidence'][0]
                    parts.append(f"  - {TOPIC_LABEL[topic]}: label advises caution/contraindication in patients who have it (tier {pc['tier']}, {pc['n_positive']}/{s['n_labels']} labels)\n    > \"{e['text'][:260]}\"" + (f" ({e['url']})" if e.get('url') else ''))
            L.append(f"- **{ing}**" + (':' if parts else ': none found in the available FDA labels for QT, slow heart rate or heart block (not evidence of safety)'))
            L += parts
    L += ['', '> ' + DISCLAIMER]
    return '\n'.join(L)


In [ ]:
%%writefile /content/drive/MyDrive/CardioSafe/cardiosafe_rx.py
"""Prescription pipeline: synthetic prescription images, OCR wrappers, drug-line extraction, scoring.
Extraction output goes through cardiosafe_kb.resolve_drug; anything not trusted stays 'unrecognised' (never guessed)."""
import re, random
import numpy as np

# ------------------------------------------------------------------ line parsing
FORMS = r'(?:tab(?:let)?s?|cap(?:sule)?s?|syp|syrup|susp(?:ension)?|inj(?:ection)?|drops?|oint(?:ment)?|cream|gel|inh(?:aler)?|tabs?|caps?)'
STRENGTH = re.compile(r'\b\d+(?:\.\d+)?\s*(?:mg|mcg|µg|ug|g|ml|iu|units?)(?:\s*/\s*\d+(?:\.\d+)?\s*(?:mg|ml|g))?', re.I)
FREQ = re.compile(r'\b(?:OD|BD|BID|TDS|TID|QID|HS|SOS|PRN|QD|STAT|once daily|twice daily|thrice daily|at night|morning|\d\s*-\s*\d\s*-\s*\d(?:\s*-\s*\d)?)\b', re.I)
DUR = re.compile(r'(?:x|for)\s*\d+\s*(?:days?|d|weeks?|wks?|months?)\b', re.I)
SKIP = re.compile(r'^(?:dr\b|dr\.|name|patient|age|sex|date|address|clinic|hospital|reg|phone|ph\b|advice|follow|review|signature|sign\b|diagnosis|dx\b|c/o|bp\b|wt\b|weight|rx\s*$|℞\s*$)', re.I)
PREFIX = re.compile(r'^\s*(?:\d{1,2}\s*[.)\]:-]\s*)?(?:rx\s*[:.\-]?\s*)?(?:' + FORMS + r'\.?\s+)?', re.I)


def parse_line(line):
    """-> dict(raw, name, strength, frequency, duration) or None if the line does not look like a medication line."""
    raw = line.strip()
    if len(raw) < 3 or SKIP.match(raw):
        return None
    has_form = bool(re.match(r'^\s*(?:\d{1,2}\s*[.)\]:-]\s*)?(?:rx\s*[:.\-]?\s*)?' + FORMS + r'\b', raw, re.I))
    st, fq, du = STRENGTH.search(raw), FREQ.search(raw), DUR.search(raw)
    if not (has_form or st or fq):
        return None
    body = PREFIX.sub('', raw, count=1)
    cut = len(body)
    for m in (STRENGTH.search(body), FREQ.search(body), DUR.search(body), re.search(r'\b\d', body)):
        if m:
            cut = min(cut, m.start())
    name = re.sub(r'[^A-Za-z0-9 \-]', ' ', body[:cut]).strip()
    name = re.sub(r'\s+', ' ', name)
    if len(name) < 3:
        return None
    return {'raw': raw, 'name': name, 'strength': st.group(0).strip() if st else None,
            'frequency': fq.group(0).strip() if fq else None, 'duration': du.group(0).strip() if du else None}


def extract_prescription(text, resolve):
    """text -> list of medication dicts, each with resolver output. `resolve(name) -> dict(status, ingredients, route, ...)`."""
    meds = []
    for line in text.splitlines():
        p = parse_line(line)
        if p is None:
            continue
        r = resolve(p['name'])
        if r['status'] != 'ok':                                                   # form word glued to the name by OCR ('TabOndem')
            g = re.match(r'^(?:tab|tap|tah|cap|syp|inj)(?=[A-Za-z]{4,})(.+)$', p['name'], re.I)
            if g:
                r2 = resolve(g.group(1))
                if r2['status'] == 'ok':
                    r = {**r2, 'route': str(r2.get('route')) + '+split_glued_prefix'}
        words = p['name'].split()
        if r['status'] != 'ok' and len(words) >= 2 and len(words[0]) <= 4:       # OCR-garbled form prefix such as 'Tub' / 'Tab,'
            r2 = resolve(' '.join(words[1:]))
            if r2['status'] == 'ok':
                r = {**r2, 'route': str(r2.get('route')) + '+dropped_leading_token'}
        p.update(status=r['status'], route=r.get('route'), ingredients=list(r.get('ingredients') or []), similarity=r.get('similarity'))
        meds.append(p)
    return meds


# ------------------------------------------------------------------ OCR wrappers
def ocr_tesseract(img, psm=6):
    import pytesseract
    g = img.convert('L')
    if g.width < 1400:                                   # Tesseract works best with letter height of about 30 px
        from PIL import Image
        f = 1400 / g.width
        g = g.resize((int(g.width * f), int(g.height * f)), Image.LANCZOS)
    return pytesseract.image_to_string(g, config=f'--oem 1 --psm {psm}')


_EASY = {}
def ocr_easyocr(img):
    import easyocr
    if 'r' not in _EASY:
        _EASY['r'] = easyocr.Reader(['en'], gpu=False, verbose=False)
    res = _EASY['r'].readtext(np.array(img.convert('RGB')), detail=1, paragraph=False)
    # group boxes into lines by vertical centre
    items = sorted(((np.mean([p[1] for p in b]), min(p[0] for p in b), t) for b, t, c in res))
    lines, cur, last_y = [], [], None
    for y, x, t in items:
        if last_y is not None and abs(y - last_y) > 18:
            lines.append(' '.join(w for _, w in sorted(cur))); cur = []
        cur.append((x, t)); last_y = y
    if cur:
        lines.append(' '.join(w for _, w in sorted(cur)))
    return '\n'.join(lines)


# ------------------------------------------------------------------ synthetic prescriptions
DOCTORS = ['Dr. A. Sharma', 'Dr. R. Iyer', 'Dr. S. Khan', 'Dr. P. Menon', 'Dr. L. Fernandes']
PATIENTS = ['Rohan Mehta', 'Anita Das', 'Imran Qureshi', 'Meera Nair', 'John Dsouza', 'Kavita Rao']
FREQS = ['OD', 'BD', 'TDS', 'HS', '1-0-1', '1-1-1', '0-0-1', 'SOS']
DURS = ['x 3 days', 'x 5 days', 'x 7 days', 'x 10 days', 'x 1 month', '']
GENERIC_STRENGTHS = {
    'azithromycin': '500 mg', 'ciprofloxacin': '500 mg', 'levofloxacin': '500 mg', 'ondansetron': '4 mg', 'metoprolol': '50 mg', 'atenolol': '50 mg',
    'amiodarone': '200 mg', 'amlodipine': '5 mg', 'atorvastatin': '10 mg', 'metformin': '500 mg', 'citalopram': '20 mg', 'escitalopram': '10 mg',
    'haloperidol': '5 mg', 'quetiapine': '25 mg', 'donepezil': '5 mg', 'hydroxychloroquine': '200 mg', 'fluconazole': '150 mg', 'digoxin': '0.25 mg',
    'diltiazem': '30 mg', 'verapamil': '80 mg', 'propranolol': '40 mg', 'lisinopril': '5 mg', 'levothyroxine': '50 mcg', 'pantoprazole': '40 mg',
    'omeprazole': '20 mg', 'ramipril': '5 mg', 'telmisartan': '40 mg', 'bisoprolol': '5 mg', 'rosuvastatin': '10 mg', 'clarithromycin': '500 mg',
    'moxifloxacin': '400 mg', 'methadone': '10 mg', 'sotalol': '80 mg', 'chlorpromazine': '25 mg', 'ibuprofen': '400 mg', 'aspirin': '75 mg',
}
GENERIC_ALIASES = {'acetaminophen': ['paracetamol', 'acetaminophen']}


def build_pool(brand_rows):
    """brand_rows: list of (brand, 'ing1;ing2', origin). Returns list of dict(name, truth(set), kind, strength)."""
    pool = []
    for b, ing, _ in brand_rows:
        ings = [i.strip() for i in ing.split(';')]
        if len(ings) == 1 and ings[0] in GENERIC_STRENGTHS or ings[0] == 'paracetamol':
            st = GENERIC_STRENGTHS.get(ings[0], '500 mg')
            pool.append({'name': b, 'truth': {'acetaminophen' if ings[0] == 'paracetamol' else ings[0]}, 'kind': 'brand', 'strength': st})
        elif len(ings) == 2:
            pool.append({'name': b, 'truth': set(ings), 'kind': 'brand', 'strength': '625 mg'})
    for g, st in GENERIC_STRENGTHS.items():
        pool.append({'name': g, 'truth': {g}, 'kind': 'generic', 'strength': st})
    pool.append({'name': 'paracetamol', 'truth': {'acetaminophen'}, 'kind': 'generic', 'strength': '500 mg'})
    return pool


def make_prescription(pool, rng, n_meds=None, decoys=0):
    """-> (lines, truth: list of dict(name, truth set)). `decoys` adds non-drug lines (advice etc.)."""
    n = n_meds or rng.integers(2, 6)
    picks = [pool[i] for i in rng.choice(len(pool), n, replace=False)]
    lines = [rng.choice(DOCTORS) + ', MBBS, MD', 'City Care Clinic, Reg. No. 48213', f'Name: {rng.choice(PATIENTS)}   Age: {rng.integers(25, 85)} / {rng.choice(["M", "F"])}',
             f'Date: {rng.integers(1, 28):02d}/{rng.integers(1, 12):02d}/2026', 'Rx', '']
    truth = []
    for i, m in enumerate(picks, 1):
        form = rng.choice(['Tab', 'Tab', 'Cap', 'Tab.'])
        name = m['name'].capitalize() if m['kind'] == 'brand' else m['name'].lower()
        fr, du = rng.choice(FREQS), rng.choice(DURS)
        style = rng.integers(0, 3)
        line = f'{i}. {form} {name} {m["strength"]} {fr} {du}'.strip() if style else f'{i}) {form} {name} {fr} {du}'.strip()
        lines.append(line); truth.append({'name': m['name'], 'truth': m['truth']})
    lines += ['', 'Advice: plenty of fluids, rest', 'Review after 1 week']
    return lines, truth


def render(lines, rng, level='easy', size=None):
    """Render lines to a PIL image. level: easy (clean scan) / medium (rotation+noise+blur) / hard (strong degradation)."""
    from PIL import Image, ImageDraw, ImageFont, ImageFilter
    import os, matplotlib
    ttf = os.path.join(matplotlib.get_data_path(), 'fonts', 'ttf')          # bundled with matplotlib, so always present (Colab has no system DejaVu)
    fonts = [os.path.join(ttf, f) for f in ('DejaVuSans.ttf', 'DejaVuSerif.ttf', 'DejaVuSansMono.ttf') if os.path.exists(os.path.join(ttf, f))]
    if not fonts:
        raise RuntimeError('no TrueType font found; refusing to render with the tiny default font')
    font = ImageFont.truetype(str(rng.choice(fonts)), size or int(rng.integers(26, 34)))
    W, lh = 1100, 52
    img = Image.new('RGB', (W, lh * (len(lines) + 2)), (int(rng.integers(235, 255)),) * 3)
    d = ImageDraw.Draw(img)
    for i, t in enumerate(lines):
        d.text((50, 30 + i * lh), t, fill=(20, 20, 30), font=font)
    if level == 'easy':
        return img
    ang = float(rng.uniform(-2.5, 2.5) if level == 'medium' else rng.uniform(-5, 5))
    img = img.rotate(ang, expand=True, fillcolor=(245, 245, 245))
    a = np.asarray(img).astype(np.float32)
    sig = 12 if level == 'medium' else 30
    a += rng.normal(0, sig, a.shape)
    a = np.clip(a, 0, 255).astype(np.uint8)
    img = Image.fromarray(a).filter(ImageFilter.GaussianBlur(0.8 if level == 'medium' else 1.6))
    if level == 'hard':
        w, h = img.size
        img = img.resize((int(w * 0.55), int(h * 0.55)))
    import io
    buf = io.BytesIO(); img.save(buf, 'JPEG', quality=70 if level == 'medium' else 35); buf.seek(0)
    return Image.open(buf).convert('RGB')


# ------------------------------------------------------------------ scoring
def score_prescriptions(preds, truths):
    """preds: list of med lists (extract_prescription output); truths: list of list(dict(truth set)).
    Drug-level (ingredient) precision / recall; a med counts as correct when its resolved ingredient set equals the truth set
    of one prescribed drug. Unrecognised meds count as missed (recall), not as false positives."""
    tp = fp = fn = unrec = 0
    for meds, tr in zip(preds, truths):
        gt = [frozenset(t['truth']) for t in tr]
        got = [frozenset(m['ingredients']) for m in meds if m['status'] == 'ok' and m['ingredients']]
        unrec += sum(m['status'] != 'ok' for m in meds)
        rem = list(gt)
        for g in got:
            if g in rem:
                tp += 1; rem.remove(g)
            else:
                fp += 1
        fn += len(rem)
    p, r = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {'precision': p, 'recall': r, 'f1': 2 * p * r / max(p + r, 1e-9), 'TP': tp, 'FP': fp, 'FN': fn, 'unrecognised_lines': unrec}


In [ ]:
%%writefile /content/drive/MyDrive/CardioSafe/cardiosafe_app.py
"""CardioSafe AI - Gradio demo. Three modes: ECG only, prescription only, ECG + prescription.
Run:  CARDIOSAFE_WORK=/path/to/project python app.py   (or call build_app(work).launch())"""
import os, sys, json, glob
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gradio as gr

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import cardiosafe_common as C
import cardiosafe_infer as I
import cardiosafe_combine as CB
import cardiosafe_explain as EX
import cardiosafe_kb as K
import cardiosafe_rx as R
import torch

NONE_ECG = '(no ECG - prescription only)'
EXAMPLE_RX = {
    '(type or upload your own)': '',
    'QT-risk set': 'Rx\n1. Tab Azithral 500 mg OD x 5 days\n2. Tab Ondem 4 mg SOS\n3. Tab Dolo 650 mg TDS x 3 days',
    'Rate-slowing set': 'Rx\n1. Tab Metolar 50 mg OD\n2. Tab Dilzem 30 mg TDS\n3. Tab Storvas 10 mg HS',
    'Benign set': 'Rx\n1. Tab Glycomet 500 mg BD\n2. Tab Telma 40 mg OD\n3. Tab Pantocid 40 mg OD',
    'Includes an unknown / combination brand': 'Rx\n1. Tab Ciplox 500 mg BD x 7 days\n2. Tab Ecosprin AV 75 mg HS\n3. Tab Xyzzyflam 10 mg OD',
}


class App:
    def __init__(self, work):
        self.work = work
        self.engine = None
        kb = os.path.join(work, 'kb')
        self.fda_cache = K.JsonCache(os.path.join(kb, 'cache_openfda.json'))
        self.rx_cache = K.JsonCache(os.path.join(kb, 'cache_rxnorm.json'))
        self.bm = K.load_brand_map(os.path.join(kb, 'brand_map.csv'))
        self.api_key = os.environ.get('OPENFDA_API_KEY')
        z = np.load(os.path.join(work, 'demo_examples.npz'), allow_pickle=True)
        self.X, self.sex, self.tags, self.truth, self.ids = z['X'], z['sex'], list(z['tags']), list(z['truth']), z['ecg_id']
        self.choices = [NONE_ECG] + [f'#{i} | ECG {self.ids[i]} | {self.tags[i]} | dataset label: {self.truth[i]}' for i in range(len(self.X))]
        self.rx_imgs = sorted(glob.glob(os.path.join(work, 'demo', 'rx_example_*.jpg')))

    def eng(self):
        if self.engine is None:
            self.engine = I.ECGEngine(self.work)
        return self.engine

    def fetch(self, ing):
        return K.fda_fetch(ing, self.fda_cache, self.api_key)

    def resolve(self, name):
        return K.resolve_drug(name, self.bm, self.rx_cache)

    # ---- steps
    def read_image(self, img):
        if img is None:
            return ''
        return R.ocr_tesseract(img, 4)

    def load_ecg(self, choice, upload, fs, sex_choice):
        if upload:
            path = upload if isinstance(upload, str) else upload.name
            a = np.load(path) if path.endswith('.npy') else np.loadtxt(path, delimiter=',')
            return I.prepare_signal(a, fs=int(fs)), 0 if sex_choice == 'Male' else 1, 'uploaded file'
        i = int(choice.split('|')[0].strip().lstrip('#'))
        return self.X[i], int(self.sex[i]), f'example {self.ids[i]}'

    def plot(self, sig, ecg, cam=None):
        fig, axes = plt.subplots(6, 2, figsize=(11, 8), sharex=True)
        t = np.arange(sig.shape[0]) / 100
        for k, a in enumerate(axes.T.flatten()):
            a.plot(t, sig[:, k], lw=.7, color='#1f3a5f')
            a.set_ylabel(C.LEADS[k], rotation=0, labelpad=14, fontsize=8); a.set_yticks([]); a.margins(x=0)
            if cam is not None:
                a.fill_between(t, sig[:, k].min(), sig[:, k].max(), where=cam > 0.6, color='#e15759', alpha=.18, lw=0)
        axes[-1, 0].set_xlabel('seconds'); axes[-1, 1].set_xlabel('seconds')
        fig.suptitle(f"QTc {ecg['qtc_fridericia_ms']:.0f} ms ({ecg['qtc_status']})  |  HR {ecg['hr_bpm']:.0f} bpm  |  red = regions the model relied on for its top class (Grad-CAM, explanation not proof)", fontsize=9)
        fig.tight_layout()
        return fig

    def cam_for(self, sig, ecg):
        try:
            E = self.eng(); net = E.members[0]
            x = C.to_tensor_ncl(sig[None], E.mu, E.sd, E.dev)
            top = max(ecg['probs'], key=ecg['probs'].get); cls = C.CLASSES.index(top)
            return EX.gradcam(net, net.body[-1], x, cls)[0]
        except Exception:
            return None

    def run(self, choice, upload, fs, sex_choice, rx_text):
        rx_text = (rx_text or '').strip()
        have_ecg = bool(upload) or (choice and choice != NONE_ECG)
        if not have_ecg and not rx_text:
            return None, 'Choose an ECG example or upload one, and/or add a prescription.', '{}'
        meds = R.extract_prescription(rx_text, self.resolve) if rx_text else []
        self.rx_cache.save()
        if not have_ecg:
            rep = CB.drug_only_report(meds, self.fetch) if meds else 'No medication lines could be read from the prescription text.'
            self.fda_cache.save()
            return None, rep, json.dumps(meds, indent=1, default=str)
        try:
            sig, sex, src = self.load_ecg(choice, upload, fs, sex_choice)
        except Exception as e:
            return None, f'Could not read the ECG file: {e}', '{}'
        ecg = self.eng().analyze(sig, sex)[0]
        fig = self.plot(sig, ecg, self.cam_for(sig, ecg))
        if not meds:
            f = CB.ecg_findings(ecg)
            rep = CB.render_report(ecg, [], None)
            return fig, rep, json.dumps({'ecg': ecg, 'findings': f}, indent=1, default=str)
        result = CB.assess(ecg, meds, self.fetch)
        self.fda_cache.save()
        return fig, CB.render_report(ecg, meds, result), json.dumps({'ecg': ecg, 'medications': meds, 'assessment': result}, indent=1, default=str)


def build_app(work=None):
    app = App(work or os.environ.get('CARDIOSAFE_WORK', HERE))
    with gr.Blocks(title='CardioSafe AI') as demo:
        gr.Markdown('# CardioSafe AI\nECG analysis + medication cardiac-safety check, with every statement tied to a source sentence. '
                    '**Research prototype - not a medical device.**')
        with gr.Row():
            with gr.Column():
                gr.Markdown('### 1. ECG')
                ecg_dd = gr.Dropdown(app.choices, value=app.choices[1] if len(app.choices) > 1 else NONE_ECG, label='PTB-XL test-set example (never seen in training)')
                up = gr.File(label='...or upload a 12-lead ECG (.npy or .csv, time x 12 leads, mV)', file_types=['.npy', '.csv'])
                with gr.Row():
                    fs = gr.Number(value=100, label='sampling rate (Hz) of uploaded file')
                    sx = gr.Radio(['Male', 'Female'], value='Male', label='sex (for uploaded file)')
            with gr.Column():
                gr.Markdown('### 2. Prescription')
                ex = gr.Dropdown(list(EXAMPLE_RX), value='QT-risk set', label='Example prescription text')
                img = gr.Image(type='pil', label='...or upload a prescription image (printed text)')
                btn_ocr = gr.Button('Read text from image (OCR)')
                rx = gr.Textbox(lines=6, label='Prescription text (edit OCR mistakes here)', value=EXAMPLE_RX['QT-risk set'])
        go = gr.Button('Analyze', variant='primary')
        plot = gr.Plot(label='ECG')
        report = gr.Markdown()
        with gr.Accordion('Raw structured output', open=False):
            raw = gr.Code(language='json')
        ex.change(lambda k: EXAMPLE_RX[k], ex, rx)
        btn_ocr.click(app.read_image, img, rx)
        go.click(app.run, [ecg_dd, up, fs, sx, rx], [plot, report, raw])
        with gr.Accordion('Limitations (read before judging the output)', open=False):
            gr.Markdown(CB.DISCLAIMER + '\n\n- ECG model trained and tested on PTB-XL only (100 Hz, 10 s, 12 leads); no external validation.\n'
                        '- QTc is a learned estimate whose training targets are algorithm outputs (12SL / Uni-G), not manual measurements.\n'
                        '- AV block / PR interval is not measured; the conduction class also contains bundle-branch blocks.\n'
                        '- OCR was tested on synthetic printed prescriptions only; handwriting is not supported. Always check the text box.\n'
                        '- Drug evidence = sentences in FDA (US) labels. Labels can omit known risks (for example some antipsychotics); "none found" is not "safe".')
    return demo


if __name__ == '__main__':
    build_app().launch()


## 1. Pick demo cases from the held-out test fold (fold 10, never used for training or tuning)

In [ ]:
import importlib, json, numpy as np, pandas as pd
import cardiosafe_common as C, cardiosafe_infer as I, cardiosafe_combine as CB, cardiosafe_kb as K, cardiosafe_rx as R
for m in (CB, R): importlib.reload(m)
X, Y, fold, tr, va, te, meta = C.load_data(WORK + '/ptbxl_100hz_super5.npz')
sex = meta['sex']; ids = meta['ecg_id']; t_idx = np.where(te)[0]
E = I.ECGEngine(WORK)
res = []
for i in range(0, len(t_idx), 256): res += E.analyze(X[t_idx[i:i + 256]], sex[t_idx[i:i + 256]])
def pick(cond, n): return [k for k, r in enumerate(res) if cond(r)][:n]
sel, tags = [], {}
for tag, cond, n in [('QTc prolonged', lambda r: r['qtc_status'] == 'prolonged' and r['qt_seed_sd_ms'] < 15, 4),
                     ('QTc borderline', lambda r: r['qtc_status'] == 'borderline' and r['qt_seed_sd_ms'] < 15, 3),
                     ('slow heart rate (<50 bpm)', lambda r: r['hr_bpm'] < 50, 3),
                     ('conduction disturbance (CD) confident', lambda r: r['conformal']['CD'] == 'confident_positive', 3),
                     ('normal-looking', lambda r: r['conformal']['NORM'] == 'confident_positive' and r['qtc_status'] == 'normal', 3),
                     ('model uncertain', lambda r: sum(v == 'uncertain' for v in r['conformal'].values()) >= 2, 2)]:
    for k in pick(cond, n * 3):
        if k not in tags and sum(t == tag for t in tags.values()) < n: tags[k] = tag
sel = list(tags)
print(pd.Series(list(tags.values())).value_counts().to_string())
np.savez(WORK + '/demo_examples.npz', X=X[t_idx[sel]], sex=sex[t_idx[sel]], ecg_id=ids[t_idx[sel]], tags=np.array([tags[k] for k in sel]),
         truth=np.array([','.join(c for c, v in zip(C.CLASSES, Y[t_idx[k]]) if v) or 'none' for k in sel]))
print('saved demo_examples.npz with', len(sel), 'ECGs')

In [ ]:
# three printed example prescriptions (synthetic) for the OCR demo
bm = K.load_brand_map(KB + '/brand_map.csv'); import pandas as pd
pool = R.build_pool([(r.brand, r.ingredients, r.origin) for r in pd.read_csv(KB + '/brand_map.csv').itertuples()])
texts = [['Dr. A. Sharma, MBBS, MD', 'City Care Clinic, Reg. No. 48213', 'Name: Rohan Mehta   Age: 61 / M', 'Date: 12/11/2026', 'Rx', '',
          '1. Tab Azithral 500 mg OD x 5 days', '2. Tab Ondem 4 mg SOS', '3. Tab Dolo 650 mg TDS x 3 days', '', 'Advice: plenty of fluids, rest'],
         ['Dr. P. Menon, MBBS', 'City Care Clinic, Reg. No. 48213', 'Name: Meera Nair   Age: 70 / F', 'Date: 14/11/2026', 'Rx', '',
          '1. Tab Metolar 50 mg OD', '2. Tab Storvas 10 mg HS', '3. Tab Telma 40 mg OD', '', 'Review after 2 weeks'],
         ['Dr. S. Khan, MBBS', 'City Care Clinic, Reg. No. 48213', 'Name: Imran Qureshi   Age: 48 / M', 'Date: 15/11/2026', 'Rx', '',
          '1. Tab Ciplox 500 mg BD x 7 days', '2. Tab Ecosprin AV 75 mg HS', '3. Tab Pantocid 40 mg OD', '', 'Advice: take after food']]
for k, ln in enumerate(texts, 1):
    R.render(ln, np.random.default_rng(k), 'medium').save(f'{WORK}/demo/rx_example_{k}.jpg')
print('saved 3 example prescription images')

## 2. Check the app logic on the three modes before launching

In [ ]:
import cardiosafe_app as A
importlib.reload(A)
app = A.App(WORK)
print(len(app.choices) - 1, 'ECG examples loaded')
ci = next((i for i, c in enumerate(app.choices) if 'QTc prolonged' in c), 1)
for name, ch, rxt in [('ECG + prescription', app.choices[ci], A.EXAMPLE_RX['QT-risk set']), ('prescription only', A.NONE_ECG, A.EXAMPLE_RX['Rate-slowing set']), ('ECG only', app.choices[ci], '')]:
    fig, rep, raw = app.run(ch, None, 100, 'Male', rxt)
    print('=' * 90, '\n', name, '->', rep[:700].replace('\n', '\n   '))
    if fig is not None:
        import matplotlib.pyplot as plt; plt.close(fig)
print('OCR check on example image 1:\n', app.read_image(__import__('PIL.Image', fromlist=['Image']).open(WORK + '/demo/rx_example_1.jpg'))[:400])

## 3. Package the Hugging Face Space (so the demo does not depend on Colab)

In [ ]:
import shutil, json, zipfile
SP = '/content/cardiosafe_space'; shutil.rmtree(SP, ignore_errors=True); os.makedirs(SP + '/models'); os.makedirs(SP + '/kb'); os.makedirs(SP + '/demo')
trust = json.load(open(WORK + '/trust_config.json'))
for m in trust['members']: shutil.copy(f'{WORK}/models/{m}.pt', SP + '/models/')
for s in range(3): shutil.copy(f'{WORK}/models/qtnet_s{s}.pt', SP + '/models/')
for f in ['trust_config.json', 'trust_artifacts.npz', 'norm_stats.npz', 'qtc_config_learned.json', 'demo_examples.npz', 'cardiosafe_common.py', 'cardiosafe_models.py',
          'cardiosafe_trust.py', 'cardiosafe_qtnet.py', 'cardiosafe_explain.py', 'cardiosafe_infer.py', 'cardiosafe_kb.py', 'cardiosafe_rx.py', 'cardiosafe_combine.py', 'cardiosafe_app.py']:
    shutil.copy(f'{WORK}/{f}', SP + '/')
for f in ['brand_map.csv', 'cache_openfda.json', 'cache_rxnorm.json']: shutil.copy(f'{KB}/{f}', SP + '/kb/')
for f in os.listdir(WORK + '/demo'): shutil.copy(f'{WORK}/demo/{f}', SP + '/demo/')
open(SP + '/app.py', 'w').write("import os\nfrom cardiosafe_app import build_app\ndemo = build_app(os.path.dirname(os.path.abspath(__file__)))\nif __name__ == '__main__':\n    demo.launch()\n")
open(SP + '/requirements.txt', 'w').write('torch\nnumpy\nscipy\nscikit-learn\nmatplotlib\npandas\nrapidfuzz\npytesseract\nrequests\npillow\n')
open(SP + '/packages.txt', 'w').write('tesseract-ocr\n')
open(SP + '/README.md', 'w').write('---\ntitle: CardioSafe AI\nemoji: heart\ncolorFrom: red\ncolorTo: blue\nsdk: gradio\napp_file: app.py\npinned: false\n---\n\nResearch prototype: ECG analysis and medication cardiac-safety check. Not a medical device.\n')
size = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(SP) for f in fs) / 1e6
print(f'space folder: {size:.0f} MB')
shutil.make_archive(WORK + '/cardiosafe_space', 'zip', SP); print('saved', WORK + '/cardiosafe_space.zip')

## 4. Launch the demo
The cell prints a public `gradio.live` link (valid while this notebook stays running). Open it in a browser tab for the presentation.

In [ ]:
demo = A.build_app(WORK)
demo.launch(share=True, debug=False)

## Presenting tips
- Start with an ECG example tagged *QTc prolonged* and the *QT-risk set* prescription -> relationship found with quoted label sentences.
- Then the *Benign set* -> "none found in available sources" (and point out it does not mean safe).
- Then *Includes an unknown / combination brand* -> the unknown drug is flagged insufficient evidence instead of guessed.
- Finish with the Limitations accordion; it is a strength of the project, not a weakness.